In [ ]:
import requests
from tqdm import tqdm
import os
import time
import logging
import shutil
from typing import *

class inaturqalistScrapper:
  def __init__(self, scientificName: str, n:int = 1000, taxon_id: Optional[str] = ""):
    '''
    '''
    self.scientificName = scientificName    # Will also be used as a folder name to store images
    self.n = n
    if not taxon_id:
      taxon_id, _ = self.get_taxon_id()
    if not taxon_id:
        raise ValueError(f'No taxon found for scientific name: {scientificName}')

    observations = self.fetch_observations(taxon_id)
    image_urls = self.extract_image_urls(observations)[::-1]
    
    print(f'Scrapped {len(image_urls)} number of image links')

    self.download_images(image_urls)

  def get_taxon_id(self):
      search_url = "https://api.inaturalist.org/v1/taxa"
      params = {
          'q': self.scientificName,
          'rank': 'species',
          'per_page': 1
      }
      try:
          response = requests.get(search_url, params=params)
          response.raise_for_status()
          data = response.json()
          if data['total_results'] > 0:
              taxon = data['results'][0]
              return taxon['id'], taxon['name']
          else:
              return None, None
      except requests.exceptions.RequestException as e:
          print(f"Error fetching taxon ID: {e}")
          return None, None

  def fetch_observations(self, taxon_id, per_page=10000):
      observations = []
      page = 1
      total_pages = 1000
      while True:
          print(f"Fetching page {page}...")
          url = "https://api.inaturalist.org/v1/observations"
          params = {
              'taxon_id': taxon_id,
              'per_page': per_page,
              'page': page,
              'order': 'desc',
              'order_by': 'created_at',
              'ident_taxon_id': taxon_id
          }
          try:
              response = requests.get(url, params=params)
              response.raise_for_status()
              data = response.json()
              results = data.get('results', [])
              meta = data.get('meta', {})
              if not results:
                  # No more observations
                  break
              observations.extend(results)
              print(f"Fetched {len(results)} observations.")
              if len(observations) >= self.n:
                  break
              if total_pages is None:
                  total_results = meta.get('total_results', 0)
                  total_pages = (total_results // per_page) + (1 if total_results % per_page != 0 else 0)
              if page > total_pages:
                  break
              page += 1
              time.sleep(1)  # Respect rate limits
          except requests.exceptions.RequestException as e:
              print(f"Failed to fetch observations: {e}")
              break
      return observations

  def extract_image_urls(self, observations):
      image_urls = []
      for obs in observations:
          photos = obs.get('photos', [])
          for photo in photos:
              url = photo.get('url', '')
              if url:
                  # Attempt to get 'original' and 'large' sizes
                  original_url = url.replace('square', 'original')
                  image_urls.append(original_url)
      # Remove duplicate URLs
      image_urls = list(set(image_urls))
      return image_urls

  def download_images(self, image_urls, save_dir="", max_retries=3):
      if not save_dir: save_dir = self.scientificName

      if not os.path.exists(save_dir):
          os.makedirs(save_dir)

      # Set up logging for failed downloads
      logging.basicConfig(filename='download_errors.log', level=logging.ERROR)

      index=0
      if self.n > len(image_urls): self.n = len(image_urls)
      for index_url in tqdm(range(self.n), desc="Downloading images"):
          filename = f'{index}.{image_urls[index_url].split(".")[-1]}'
          retries = 0
          success = False
          while retries < max_retries and not success:
              try:
                  response = requests.get(image_urls[index_url], stream=True, timeout=10)
                  if response.status_code == 200:
                      file_path = os.path.join(save_dir, filename)
                      with open(file_path, 'wb') as f:
                          for chunk in response.iter_content(1024):
                              if chunk:
                                  f.write(chunk)
                                  index+=1
                      success = True
                  else:
                      print(f"Failed to download {image_urls[index_url]}: Status code {response.status_code}")
                      retries += 1
                      time.sleep(2)  # Wait before retrying
              except requests.exceptions.RequestException as e:
                  print(f"Error downloading {image_urls[index_url]}: {e}")
                  retries += 1
                  time.sleep(2)  # Wait before retrying
          if not success:
              logging.error(f"Failed to download after {max_retries} retries: {image_urls[index_url]}")

def moveFolder(src, dst):
  shutil.copytree(src, dst, dirs_exist_ok=True)
  print(f"Moved {src} to {dst}")

def rename_files_in_subfolders(root_folder):
  """Renames files in subfolders with the format 'subfolder_name_index'."""

  for dirpath, dirnames, filenames in os.walk(root_folder):
    for i, filename in enumerate(filenames):
      subfolder_name = os.path.basename(dirpath)
      new_filename = f"{subfolder_name}_{i}{os.path.splitext(filename)[1]}"
      old_filepath = os.path.join(dirpath, filename)
      new_filepath = os.path.join(dirpath, new_filename)

      try:
        os.rename(old_filepath, new_filepath)
        print(f"Renamed '{filename}' to '{new_filename}' in '{subfolder_name}'")
      except OSError as e:
        print(f"Error renaming '{filename}' in '{subfolder_name}': {e}")

# Example usage:
root_folder_path = '/content/drive/MyDrive/Computer Vission/Object Detection/2.100.1/Data tambahan/!TAMBAHAN/JPEG_Images'  # Replace with your root folder path
rename_files_in_subfolders(root_folder_path)

In [9]:
search_url = "https://api.inaturalist.org/v1/taxa"
params = {
    'q': "kucing",
    'rank': 'species',
    'per_page': 1
}
response = requests.get(search_url, params=params)

In [10]:
response.content

b'{"total_results":9,"page":1,"per_page":1,"results":[{"id":118552,"rank":"species","rank_level":10,"iconic_taxon_id":40151,"ancestor_ids":[48460,1,2,355675,40151,848317,848320,848324,41573,41944,846179,41956,118552],"is_active":true,"name":"Felis catus","parent_id":41956,"ancestry":"48460/1/2/355675/40151/848317/848320/848324/41573/41944/846179/41956","extinct":false,"provisional":false,"default_photo":{"id":129658776,"license_code":"cc-by-sa","attribution":"(c) Von.grzanka, some rights reserved (CC BY-SA)","url":"https://inaturalist-open-data.s3.amazonaws.com/photos/129658776/square.jpg","original_dimensions":{"height":1365,"width":2048},"flags":[],"attribution_name":"Von.grzanka","square_url":"https://inaturalist-open-data.s3.amazonaws.com/photos/129658776/square.jpg","medium_url":"https://inaturalist-open-data.s3.amazonaws.com/photos/129658776/medium.jpg"},"taxon_changes_count":2,"taxon_schemes_count":3,"observations_count":161437,"flag_counts":{"resolved":10,"unresolved":0},"curre